# Tarea #9 · Elegir al jugador y no perderlo (re-identificación)

**El problema:** el tracker solo mira *dónde* está cada jugador, no *cómo es*. En `2e57b9_0.mp4` el delantero blanco (7) y el defensa que lo marca (21) se tapan en el segundo 11 y el tracker les intercambia el número.

**La solución** (el mismo método que la práctica de Face Recognition, con el cuerpo entero en vez de la cara):
1. Convertimos cada recorte de jugador en una "huella" de 512 números (*embedding*) con ResNet-18.
2. Al elegir al jugador guardamos varias huellas suyas: su **galería**.
3. En cada fotograma, de los jugadores que están cerca, nos quedamos con el que más se parece a la galería. Si ninguno se parece lo suficiente (distancia > τ), decimos "no lo veo" en vez de equivocarnos.
4. Elegimos τ con FAR y FRR, como en la práctica.

**Antes de empezar:** haber ejecutado `06_homografia.ipynb` (este notebook usa sus resultados). Si se ejecutó antes de que el código guardara las cajas de los jugadores, la celda de configuración vuelve a procesar el vídeo automáticamente (unos minutos en la GPU).

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from mplsoccer import Pitch

from src.fase1_tracking import (ExtractorApariencia, calcular_embeddings, calibrar_umbral,
                                elegir_jugador_con_clic, recortar, seguir_jugador)
from src.posiciones import procesar_video, posiciones_de_un_jugador, posiciones_por_tramos
from src.fase2_homografia import LARGO_M, ANCHO_M

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

In [ ]:
# ---- Configuración ----
VIDEO = "data/clips/2e57b9_0.mp4"
SALIDA = Path("outputs") / Path(VIDEO).stem
FIG = Path("outputs/figuras"); FIG.mkdir(parents=True, exist_ok=True)

todos = pd.read_csv(SALIDA / "posiciones_todos.csv")
diag = pd.read_csv(SALIDA / "homografia.csv")
if not {"x1", "y1", "x2", "y2"} <= set(todos.columns):
    # Los resultados de 06 son anteriores a guardar las cajas: hay que procesar otra vez (unos minutos en GPU)
    print("posiciones_todos.csv no tiene las cajas de los jugadores: procesando el vídeo otra vez...")
    todos, diag = procesar_video(VIDEO, device=DEVICE, progreso=False)
    todos.to_csv(SALIDA / "posiciones_todos.csv", index=False); diag.to_csv(SALIDA / "homografia.csv", index=False)
print(f"{len(todos)} detecciones de {todos.track_id.nunique()} jugadores en {todos.frame.nunique()} fotogramas")

## 1. La huella de cada jugador (embeddings)

ResNet-18 preentrenada, **sin entrenar nada**: le quitamos la última capa (el clasificador de ImageNet) y nos quedamos con el vector de 512 valores de la penúltima, normalizado. Es el mismo "extractor fijo" de la práctica (`eval()` + `no_grad()`). Se calcula una vez y se guarda en `embeddings.npy`.

In [ ]:
ruta_embs = SALIDA / "embeddings.npy"
extractor = ExtractorApariencia(DEVICE)
if ruta_embs.exists() and len(np.load(ruta_embs)) == len(todos):
    embs = np.load(ruta_embs)
else:
    embs = calcular_embeddings(VIDEO, todos, extractor)
    np.save(ruta_embs, embs)
print("Embeddings:", embs.shape, "· norma media", np.linalg.norm(embs, axis=1).mean().round(3))

## 2. Elegir al jugador

- `USAR_CLIC = True`: se abre una ventana con el primer fotograma; haz clic sobre el jugador y pulsa una tecla.
- `USAR_CLIC = False`: se usa el número escrito en `TRACK_ID` (el delantero de este clip es el 7 al principio).

In [ ]:
USAR_CLIC = False
TRACK_ID = 7
if USAR_CLIC:
    TRACK_ID = elegir_jugador_con_clic(VIDEO, todos)
print("Jugador elegido:", TRACK_ID)

## 3. Su galería

Los embeddings de sus 5 primeros fotogramas, separados 5 fotogramas entre sí (posturas distintas). Así se ven los recortes:

In [ ]:
N_GALERIA, SEP = 5, 5
gal = todos[todos.track_id == TRACK_ID].iloc[::SEP][:N_GALERIA]
cap = cv2.VideoCapture(VIDEO)
fig, axs = plt.subplots(1, N_GALERIA, figsize=(2.2 * N_GALERIA, 3.5))
for ax, (_, r) in zip(axs, gal.iterrows()):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(r.frame)); ok, f = cap.read()
    ax.imshow(recortar(f, r.x1, r.y1, r.x2, r.y2, margen=0.15)); ax.set_title(f"fotograma {r.frame}"); ax.axis("off")
cap.release(); fig.suptitle(f"Galería del jugador {TRACK_ID}", y=1.04)
plt.savefig(FIG / "reid_galeria.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Elegir el umbral τ con FAR y FRR

Necesitamos saber, en algunos fotogramas, quién es de verdad el delantero. En este clip lo sabemos porque lo comprobamos a mano en `06`: es el **7 hasta el fotograma 281** y el **21 desde el 288**.

- **Mismo jugador:** distancia de sus recortes a la galería → deberían ser pequeñas.
- **Jugadores que se le acercan (a 1–3 m):** son los que de verdad se pueden confundir con él → deberían ser grandes.

¿Por qué no comparar con *todos* los demás? Porque sus compañeros llevan la misma camiseta y, con jugadores de ~20 × 50 píxeles, son indistinguibles por apariencia. No hace falta distinguirlos: el seguidor solo decide por apariencia entre los que están **pegados** a él, y para lo demás confía en el tracker y en la distancia que puede recorrer.

Como en la práctica: **elegimos τ con la primera mitad del clip (validation)** y **comprobamos con la segunda (test)**, sin volver a tocarlo.

In [ ]:
TRAMOS_REALES = [(7, None, 281), (21, 288, None)]     # comprobado a mano (solo para este clip)
es_el = np.zeros(len(todos), bool)
for tid, ini, fin in TRAMOS_REALES:
    es_el |= (todos.track_id == tid) & (todos.frame >= (ini or 0)) & (todos.frame <= (fin or 10**9))
es_el = es_el.to_numpy()

real = posiciones_por_tramos(todos, diag, TRAMOS_REALES).set_index("frame")   # dónde está él (metros)
dist_a_el = np.hypot(todos.x_m - todos.frame.map(real.x_m), todos.y_m - todos.frame.map(real.y_m)).to_numpy()
cerca = (dist_a_el >= 1.0) & (dist_a_el < 3.0) & ~es_el      # los que se le acercan

galeria = embs[gal.index.to_numpy()]
d = np.linalg.norm(embs[:, None, :] - galeria[None], axis=2).min(axis=1)
no_galeria = ~np.isin(np.arange(len(todos)), gal.index)
mitad = todos.frame.median()
val = (todos.frame < mitad).to_numpy() & no_galeria
test = (todos.frame >= mitad).to_numpy() & no_galeria

TAU, tabla = calibrar_umbral(d[val & es_el], d[val & cerca])
far_t = (d[test & cerca] <= TAU).mean(); frr_t = (d[test & es_el] > TAU).mean()
print(f"τ elegido en validation: {TAU:.3f}")
print(f"En test → FAR {far_t:.2%} (otros aceptados por error) · FRR {frr_t:.2%} (él rechazado por error)")
print(f"Recortes: {(val & es_el).sum()} suyos y {(val & cerca).sum()} de jugadores cercanos en validation")

fig, axs = plt.subplots(1, 2, figsize=(14, 4))
axs[0].hist(d[val & es_el], bins=40, alpha=.7, label="mismo jugador", density=True)
axs[0].hist(d[val & cerca], bins=40, alpha=.7, label="jugadores a 1-3 m de él", density=True)
axs[0].axvline(TAU, color="k", ls="--", label=f"τ = {TAU:.2f}"); axs[0].set_xlabel("distancia a la galería"); axs[0].legend()
axs[0].set_title("Distancias en validation")
axs[1].plot(tabla.tau, tabla.FAR, label="FAR"); axs[1].plot(tabla.tau, tabla.FRR, label="FRR")
axs[1].axvline(TAU, color="k", ls="--"); axs[1].set_xlabel("umbral τ"); axs[1].legend(); axs[1].set_title("FAR y FRR según el umbral")
plt.savefig(FIG / "reid_umbral.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Seguir al jugador con la re-identificación

En cada fotograma:
- Si su número sigue ahí, sin nadie pegado y parecido a la galería → confiamos en el tracker.
- Si otro jugador está a menos de 2 m (posible cambio de número) o su número desaparece → gana el más parecido a la galería entre los que están a distancia alcanzable corriendo (≤ 10 m/s), si baja de τ.
- Si nadie baja de τ → "no localizado" en ese fotograma.

In [ ]:
seg = seguir_jugador(todos, embs, TRACK_ID, TAU, n_galeria=N_GALERIA, sep_galeria=SEP)
print(f"Fotogramas con el jugador localizado: {seg.valido.mean():.1%}")
for _, r in seg[seg.cambio_id == 1].iterrows():
    print(f"  fotograma {int(r.frame)}: el tracker le cambió el número → ahora es el {int(r.track_id)} (distancia {r.distancia:.2f})")

# ¿Coincide con lo que comprobamos a mano?
comp = seg.merge(real[["x_m", "y_m", "valido"]].reset_index(), on="frame", suffixes=("", "_real"))
ambos = (comp.valido == 1) & (comp.valido_real == 1)
error_m = np.hypot(comp.x_m - comp.x_m_real, comp.y_m - comp.y_m_real)[ambos]
se_ve = comp.valido_real == 1
print(f"Localizado en el {comp.valido[se_ve].mean():.1%} de los fotogramas en que se le ve")
print(f"De esos, es el delantero de verdad (< 1 m) en el {(error_m < 1).mean():.1%}; se va a OTRO jugador en {(error_m >= 1).sum()} fotogramas")

fig, ax = plt.subplots(figsize=(14, 4))
sin_gal = seg.distancia > 0     # los fotogramas de la propia galería tienen distancia 0
ax.plot(seg.t_s[sin_gal], seg.distancia[sin_gal], ".", ms=3, label="distancia del elegido a la galería")
ax.axhline(TAU, color="r", ls="--", label=f"τ = {TAU:.2f}")
for _, r in seg[seg.cambio_id == 1].iterrows():
    ax.axvline(r.t_s, color="orange", lw=2, label="cambio de número detectado")
perdido = seg.valido == 0
ax.fill_between(seg.t_s, 0, 1, where=perdido, transform=ax.get_xaxis_transform(), color="grey", alpha=.3, label="no localizado")
ax.set_xlabel("tiempo (s)"); ax.set_ylabel("distancia"); ax.legend(loc="upper right")
ax.set_title("Re-identificación del jugador a lo largo del clip")
plt.savefig(FIG / "reid_distancia_tiempo.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Vídeo de control: el jugador seguido en verde

In [ ]:
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS); w = int(cap.get(3)); h = int(cap.get(4))
out = cv2.VideoWriter(str(SALIDA / "control_reid.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
por_frame = todos.groupby("frame"); elegido = seg.set_index("frame")
n = 0
while True:
    ok, f = cap.read()
    if not ok: break
    if n in por_frame.groups:
        for _, r in por_frame.get_group(n).iterrows():
            nuestro = n in elegido.index and elegido.loc[n, "track_id"] == r.track_id
            color = (0, 255, 0) if nuestro else (255, 255, 255)
            cv2.rectangle(f, (r.x1, r.y1), (r.x2, r.y2), color, 3 if nuestro else 1)
            cv2.putText(f, str(r.track_id), (r.x1, r.y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    estado = "localizado" if n in elegido.index and elegido.loc[n, "valido"] == 1 else "NO LOCALIZADO"
    cv2.putText(f, f"Jugador seguido: {estado}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)
    out.write(f); n += 1
cap.release(); out.release()
print("Vídeo guardado en", SALIDA / "control_reid.mp4")

## 7. `posiciones.csv` del jugador (ya sin arreglos a mano)

In [ ]:
ATACA_IZQUIERDA = True    # los blancos atacan hacia la izquierda de la cámara
pos = seg[["frame", "t_s", "x_m", "y_m", "valido"]].copy()
pos = diag[["frame", "t_s"]].merge(pos.drop(columns="t_s"), on="frame", how="left")
pos["valido"] = pos.valido.fillna(0).astype(int)
pos.insert(2, "track_id", TRACK_ID); pos.insert(3, "equipo", "")
if ATACA_IZQUIERDA:
    pos["x_m"] = (LARGO_M - pos.x_m).round(2); pos["y_m"] = (ANCHO_M - pos.y_m).round(2)
pos.to_csv(SALIDA / "posiciones.csv", index=False)
print(f"{SALIDA / 'posiciones.csv'}: {pos.valido.mean():.1%} de fotogramas válidos")

pitch = Pitch(pitch_type="custom", pitch_length=LARGO_M, pitch_width=ANCHO_M, line_color="white", pitch_color="#3a7d44")
fig, ax = pitch.draw(figsize=(10, 7)); v = pos[pos.valido == 1]
pitch.plot(v.x_m, v.y_m, ax=ax, color="orange", lw=2)
pitch.scatter(v.x_m.iloc[:1], v.y_m.iloc[:1], ax=ax, s=150, c="white", edgecolors="k", zorder=3, label="inicio")
pitch.scatter(v.x_m.iloc[-1:], v.y_m.iloc[-1:], ax=ax, s=150, c="orange", edgecolors="k", zorder=3, label="fin")
ax.legend(loc="upper left"); ax.set_title("Trayectoria con re-identificación (atacando hacia la derecha)")
plt.savefig(FIG / "reid_trayectoria.png", dpi=150, bbox_inches="tight"); plt.show()